In [1]:
import os

In [2]:
os.environ['MINERU_API_MAX_CONCURRENT_REQUESTS']='1'

In [3]:
import pandas as pd
import asyncio
from pathlib import Path
from shutil import copy
import httpx

from mineru.cli.api_protocol import DEFAULT_MAX_CONCURRENT_REQUESTS
from mineru.utils.config_reader import get_max_concurrent_requests
from mineru.cli.client import resolve_effective_max_concurrent_requests

In [4]:
from mineru.cli.client import run_orchestrated_cli

In [5]:
df = pd.read_csv(r"D:\baseia_v3\data\inventory\sample.csv")
Path(".tmp").mkdir(exist_ok=True)

In [6]:
documentos = df['path'].to_list()

In [7]:
# for i in documentos:
#     print(Path(i).name)
#     copy(str(i), f".tmp/{Path(i).name}")

In [9]:


API_URL = "https://h5bxhwofqh2qc6-8000.proxy.runpod.net"

local_max = get_max_concurrent_requests(
    default=DEFAULT_MAX_CONCURRENT_REQUESTS,
)

async with httpx.AsyncClient(follow_redirects=True) as client:
    response = await client.get(f"{API_URL}/health")
    response.raise_for_status()
    health = response.json()

server_max = health["max_concurrent_requests"]
effective = resolve_effective_max_concurrent_requests(
    local_max=local_max,
    server_max=server_max,
)

print("env:", os.getenv("MINERU_API_MAX_CONCURRENT_REQUESTS"))
print("local_max:", local_max)
print("server_max:", server_max)
print("effective:", effective)
print("health:", health)

env: 1
local_max: 1
server_max: 1
effective: 1
health: {'status': 'healthy', 'version': '3.4.4', 'protocol_version': 2, 'queued_tasks': 0, 'processing_tasks': 0, 'completed_tasks': 0, 'failed_tasks': 0, 'max_concurrent_requests': 1, 'processing_window_size': 64, 'task_retention_seconds': 86400, 'task_cleanup_interval_seconds': 300}


In [12]:
from mineru.cli.client import run_orchestrated_cli

asyncio.run(run_orchestrated_cli(
        input_path=Path(".tmp"),
        output_dir=Path(".out"),
        method="auto",
        backend="pipeline",
        server_url=None,
        api_url="https://h5bxhwofqh2qc6-8000.proxy.runpod.net",
        lang="ch",
        start_page_id=0,
        end_page_id=None,
        formula_enable=True,
        table_enable=True,
        image_analysis=True,
    ))

2026-07-31 05:51:13.061 | INFO     | mineru.cli.client:run_planned_task:832 - Submitting batch 1/8 | 1 document, 909 pages in this batch | 4461 pages total | task#1 [2015_Book_StatisticalAnalysisAndDataDisp]


CancelledError: 

2026-07-31 05:51:27.400 | INFO     | mineru.cli.client:run_planned_task:832 - Submitting batch 2/8 | 1 document, 589 pages in this batch | 4461 pages total | task#2 [The Definitive Guide to DAX-Business intelligence with Microsoft Excel, SQL Server Analysis Services, and Power BI]


TASK FAILED: 1 httpx ConnectTimeout ConnectTimeout('')


2026-07-31 05:51:38.492 | INFO     | mineru.cli.client:run_planned_task:832 - Submitting batch 3/8 | 1 document, 532 pages in this batch | 4461 pages total | task#3 [2014_Book_DataAnalysis]


...

TASK FAILED: 5 builtins RuntimeError RuntimeError('Cannot send a request, as the client has been closed.')


2026-07-31 05:51:56.471 | INFO     | mineru.cli.client:run_planned_task:832 - Submitting batch 7/8 | 14 documents, 506 pages in this batch | 4461 pages total | task#7 [Algorithms_on_the_Intelligent_Web, DETERMINANTS OF THE QUALITY OF FINANCIAL REPORTS; [DETERMINANTES DA QUALIDADE DOS RELATÓRIOS FINANCEIROS]; [DETERMINANTES DE LA CALIDAD DE LA INFORMAC [f666d9da], Deep distributional time series models and the probabilistic forecasting of intraday electricity prices [046af12f], +11 more]


TASK FAILED: 6 builtins RuntimeError RuntimeError('Cannot send a request, as the client has been closed.')


2026-07-31 05:52:03.315 | INFO     | mineru.cli.client:run_planned_task:832 - Submitting batch 8/8 | 59 documents, 399 pages in this batch | 4461 pages total | task#8 [GSE-29, A primer on the pricing of electric energy options in Brazil via mean-reverting stochastic processes [0c179390], GAT-26, +56 more]


TASK FAILED: 7 builtins RuntimeError RuntimeError('Cannot send a request, as the client has been closed.')
TASK FAILED: 8 builtins RuntimeError RuntimeError('Cannot send a request, as the client has been closed.')


In [11]:
import mineru.cli.client as mineru_client


original_execute = mineru_client.execute_planned_tasks


async def traced_execute_planned_tasks(
    planned_tasks,
    concurrency,
    task_runner,
):
    import asyncio

    queue = asyncio.Queue()
    failures = []

    for task in planned_tasks:
        await queue.put(task)

    for _ in range(concurrency):
        await queue.put(None)

    async def worker():
        while True:
            planned_task = await queue.get()

            try:
                if planned_task is None:
                    return

                await task_runner(planned_task)

            except Exception as exc:
                print(
                    "TASK FAILED:",
                    planned_task.index,
                    type(exc).__module__,
                    type(exc).__name__,
                    repr(exc),
                )

                failures.append(
                    mineru_client.TaskFailure(
                        task_index=planned_task.index,
                        document_stems=tuple(
                            doc.stem for doc in planned_task.documents
                        ),
                        message=f"{type(exc).__name__}: {exc!r}",
                    )
                )

            finally:
                queue.task_done()

    workers = [
        asyncio.create_task(worker())
        for _ in range(concurrency)
    ]

    await queue.join()
    await asyncio.gather(*workers, return_exceptions=True)

    return failures


mineru_client.execute_planned_tasks = traced_execute_planned_tasks